# 02. Когортный retention и проверка гипотезы о первом чеке

Центральный ноутбук проекта. Здесь решается основная аналитическая задача: что отличает клиентов, совершающих повторные покупки, от тех, кто ограничивается единственной транзакцией.

Применяются два метода. Выбор сделан в пользу глубины интерпретации: каждый метод доводится до конкретного бизнес-вывода с явными допущениями.

1. Когортный анализ retention. Клиенты группируются по месяцу первой покупки, затем для каждой когорты рассчитывается доля возврата по последующим месяцам. Результат визуализируется тепловой картой.
2. t-критерий Уэлча для двух независимых выборок. Проверяется гипотеза о том, что средний размер первого чека выше у клиентов, в дальнейшем совершивших повторную покупку.

Основной вывод: размер первого чека статистически значимо связан с вероятностью возврата (p-value существенно ниже 0.001). Сегмент клиентов с первым чеком выше медианы возвращается заметно чаще.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_parquet('../data/clean.parquet')
print(f'Загружено: {len(df):,} строк, {df["Customer ID"].nunique():,} клиентов')

## Часть 1. Когортный анализ retention

Логика построения. Клиенты группируются по месяцу их первой покупки (когорта). Для каждой когорты по каждому последующему месяцу рассчитывается доля клиентов, совершивших хотя бы одну покупку в этом месяце. Полученная матрица визуализируется тепловой картой, где строка соответствует когорте, столбец смещению в месяцах от когортного месяца.

Преимущества подхода. Единственная агрегированная цифра retention не отвечает на два важных вопроса: сохраняется ли уровень retention во времени и существуют ли когорты с аномальным поведением. Когортная карта позволяет увидеть оба эффекта одновременно.

In [ ]:
# Шаг 1. Определяется месяц первой покупки для каждого клиента.
df['CohortMonth'] = df.groupby('Customer ID')['InvoiceMonth'].transform('min')

# Шаг 2. Рассчитывается смещение в месяцах от когортного месяца до месяца текущей транзакции.
def months_diff(later, earlier):
    return (later.dt.year - earlier.dt.year) * 12 + (later.dt.month - earlier.dt.month)

df['CohortIndex'] = months_diff(df['InvoiceMonth'], df['CohortMonth'])

df[['Customer ID', 'InvoiceMonth', 'CohortMonth', 'CohortIndex']].head()

Назначение производных колонок:
- CohortMonth: месяц первой покупки клиента (его когорта).
- CohortIndex: число месяцев, прошедших от когортного месяца до месяца текущей транзакции. Значение 0 соответствует первой покупке, 1 следующему месяцу и так далее.

In [ ]:
# Шаг 3. Подсчёт уникальных клиентов в каждой ячейке когорта на индекс.
cohort_data = df.groupby(['CohortMonth', 'CohortIndex'])['Customer ID'].nunique().reset_index()
cohort_pivot = cohort_data.pivot(index='CohortMonth', columns='CohortIndex', values='Customer ID')

# Шаг 4. Нормировка строк на размер когорты в нулевом месяце даёт долю возврата.
cohort_size = cohort_pivot.iloc[:, 0]
retention = cohort_pivot.divide(cohort_size, axis=0) * 100

retention.round(1).head()

Интерпретация таблицы. Строка соответствует месяцу первой покупки, столбец числу прошедших месяцев. Значение в ячейке доля клиентов из соответствующей когорты, совершивших хотя бы одну покупку в данном месяце. Столбец 0 по построению равен 100%. Столбец 1 представляет retention M+1, и так далее.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    retention,
    annot=True,
    fmt='.0f',
    cmap='Blues',
    cbar_kws={'label': 'Retention, %'},
    ax=ax,
)
ax.set_title('Когортный retention, %', fontsize=14)
ax.set_xlabel('Месяцев после первой покупки')
ax.set_ylabel('Месяц первой покупки (когорта)')
ax.set_yticklabels([d.strftime('%Y-%m') for d in retention.index], rotation=0)
plt.tight_layout()
plt.savefig('../images/cohort_retention.png', dpi=120, bbox_inches='tight')
plt.show()

Наблюдения по тепловой карте:
1. Через один месяц после первой покупки возвращается порядка 22% клиентов; через шесть месяцев около 12%. Это типичная для розничного e-commerce кривая удержания: резкое падение в первый месяц с последующим пологим снижением.
2. Декабрьские когорты демонстрируют пониженный retention по сравнению с соседними месяцами. Объяснение согласуется со структурой декабрьского спроса, в которой существенна доля покупателей подарков.
3. Ранние когорты (начало 2010 года) показывают более высокий long-term retention. Это эффект времени наблюдения: у них больший временной горизонт для накопления повторных покупок.

Ключевое следствие для второй части ноутбука: основная потеря клиентов происходит в интервале между нулевым и первым месяцем (со 100% до 22%). Именно этот переход исследуется через сравнение характеристик первой покупки у двух групп клиентов.

### Усреднённая кривая retention

В дополнение к тепловой карте полезно иметь сводный график, в котором retention усреднён по всем когортам по каждому смещению.

In [ ]:
avg_retention = retention.mean(axis=0)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(avg_retention.index, avg_retention.values, marker='o', color='#4C72B0', linewidth=2)
ax.set_title('Средний retention по когортам')
ax.set_xlabel('Месяцев после первой покупки')
ax.set_ylabel('Retention, %')
ax.set_ylim(0, 100)
for x, y in zip(avg_retention.index, avg_retention.values):
    ax.annotate(f'{y:.0f}%', (x, y), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../images/avg_retention.png', dpi=120, bbox_inches='tight')
plt.show()

Сводная кривая подтверждает ключевой вывод: разрыв между однократными и возвращающимися клиентами формируется в течение первого месяца после покупки. После прохождения этого окна retention деградирует постепенно. Соответственно, любые активности по удержанию имеют максимальный потенциал именно в первые тридцать дней.

## Часть 2. Связь размера первого чека с вероятностью возврата

Содержательная гипотеза. Клиенты, в дальнейшем совершившие повторную покупку, имеют систематически более высокий размер первого чека по сравнению с однократными клиентами. Подтверждение этой гипотезы открывает практическую возможность: использовать размер первого чека как ранний предиктор LTV уже в момент совершения первой транзакции.

Формальная постановка:
- H0: средние значения первого чека в группах вернувшихся и невернувшихся клиентов равны.
- H1: средние различаются (двусторонняя альтернатива).

Уровень значимости alpha = 0.05.

In [ ]:
# Шаг 1. Определяется первый чек каждого клиента.
first_purchase = (
    df.sort_values('InvoiceDate')
    .groupby('Customer ID')
    .agg(
        first_invoice=('Invoice', 'first'),
        first_invoice_date=('InvoiceDate', 'first'),
    )
    .reset_index()
)

# Размер первого чека определяется как сумма по строкам соответствующего Invoice.
first_check = (
    df.merge(first_purchase[['Customer ID', 'first_invoice']], on='Customer ID')
    .query('Invoice == first_invoice')
    .groupby('Customer ID')['Revenue'].sum()
    .reset_index()
    .rename(columns={'Revenue': 'first_check_value'})
)

first_check.head()

In [ ]:
# Шаг 2. Разметка клиентов: returned = 1, если совершено не менее двух различных заказов.
orders_per_customer = df.groupby('Customer ID')['Invoice'].nunique().reset_index(name='n_orders')
orders_per_customer['returned'] = (orders_per_customer['n_orders'] >= 2).astype(int)

# Шаг 3. Объединение с информацией о первом чеке.
customers = first_check.merge(orders_per_customer, on='Customer ID')

share = customers['returned'].mean() * 100
print(f'Доля вернувшихся клиентов (>= 2 заказов): {share:.1f}%')
print(f'Доля однократных клиентов:                {100 - share:.1f}%')
customers.head()

Промежуточное наблюдение. Значительная часть клиентов ограничивается единственной транзакцией, что и формирует основной разрыв в воронке. Конкретное соотношение групп в данном датасете заметно смещено в сторону вернувшихся, поскольку существенная часть клиентов представляет оптовый сегмент. На массовом B2C-маркетплейсе доли распределяются иначе, однако методология анализа не зависит от конкретного баланса.

### Сравнение распределений

Перед проведением формального теста сравним распределения двух групп визуально. Боксплот удобен для одновременной оценки центральной тенденции, разброса и хвостов.

In [ ]:
# Верхний 1% значений отсечён только для целей визуализации; в t-тесте используется полная выборка.
p99 = customers['first_check_value'].quantile(0.99)
viz_data = customers[customers['first_check_value'] < p99].copy()
viz_data['Группа'] = viz_data['returned'].map({1: 'Вернулись', 0: 'Однократные'})

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=viz_data, x='Группа', y='first_check_value', ax=ax,
            palette={'Вернулись': '#55A868', 'Однократные': '#C44E52'})
ax.set_title('Размер первого чека: вернувшиеся и однократные клиенты')
ax.set_ylabel('Первый чек, £ (без верхнего 1%)')
plt.tight_layout()
plt.savefig('../images/first_check_boxplot.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
summary = customers.groupby('returned')['first_check_value'].agg(['mean', 'median', 'std', 'count']).round(2)
summary.index = ['Однократные', 'Вернулись']
summary.columns = ['Среднее, £', 'Медиана, £', 'Ст.откл, £', 'Количество']
summary

Качественные наблюдения:
1. Среднее значение первого чека у вернувшихся клиентов приблизительно вдвое превышает соответствующий показатель у однократных.
2. Медианы также различаются, что указывает на устойчивость различия не только в зоне крайних значений распределения.

Визуальные различия предварительные. Для содержательного вывода требуется формальная проверка значимости, которая выполняется ниже.

### t-критерий Уэлча

Используется t-критерий Уэлча (`equal_var=False`), а не классический t-критерий Стьюдента. Обоснование: дисперсии в двух группах заметно различаются (что подтверждается боксплотом), и тест Уэлча сохраняет корректный размер критерия в условиях гетероскедастичности.

In [ ]:
group_returned = customers.loc[customers['returned'] == 1, 'first_check_value']
group_one_time = customers.loc[customers['returned'] == 0, 'first_check_value']

t_stat, p_value = stats.ttest_ind(group_returned, group_one_time, equal_var=False)

print(f'Среднее (вернулись):    £{group_returned.mean():.2f}  (n = {len(group_returned):,})')
print(f'Среднее (однократные):  £{group_one_time.mean():.2f}  (n = {len(group_one_time):,})')
print(f'Разница средних:        £{group_returned.mean() - group_one_time.mean():.2f}')
print(f't-статистика:           {t_stat:.3f}')
print(f'p-value:                {p_value:.2e}')

### Интерпретация результата

Полученное p-value существенно ниже принятого порога значимости. Нулевая гипотеза о равенстве средних отвергается: различие в среднем размере первого чека между группами статистически значимо и не объясняется случайными колебаниями выборки.

Содержательный вывод: размер первого чека можно использовать как ранний скоринговый признак, доступный сразу после первой транзакции.

Ограничение интерпретации. Тест устанавливает наличие связи, но не направление причинности. Возможно, оба показателя являются следствием третьего фактора (например, типа клиента, источника привлечения или категории первой покупки). Доказательство причинного эффекта требует контролируемого эксперимента; соответствующий дизайн A/B-теста представлен в ноутбуке 05.

### Сегментация как практическое следствие

Для перевода результата в формат, удобный для продуктового использования, клиенты разделены на две группы по медиане размера первого чека. Сравнивается доля возврата в каждом сегменте.

In [ ]:
median_check = customers['first_check_value'].median()
customers['check_segment'] = np.where(
    customers['first_check_value'] >= median_check,
    f'Высокий чек (>= £{median_check:.0f})',
    f'Низкий чек (< £{median_check:.0f})',
)

segment_retention = customers.groupby('check_segment')['returned'].agg(['mean', 'count']).reset_index()
segment_retention.columns = ['Сегмент', 'Доля вернувшихся', 'Размер сегмента']
segment_retention['Доля вернувшихся'] = (segment_retention['Доля вернувшихся'] * 100).round(1)
segment_retention

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    segment_retention['Сегмент'],
    segment_retention['Доля вернувшихся'],
    color=['#C44E52', '#55A868'],
)
ax.set_title('Доля вернувшихся клиентов по размеру первого чека')
ax.set_ylabel('Доля вернувшихся, %')
ax.set_ylim(0, 100)
for bar, val in zip(bars, segment_retention['Доля вернувшихся']):
    ax.annotate(f'{val:.1f}%', (bar.get_x() + bar.get_width() / 2, val), ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.savefig('../images/retention_by_segment.png', dpi=120, bbox_inches='tight')
plt.show()

Сегмент с первым чеком выше медианы демонстрирует существенно более высокую долю возврата. Этот результат согласуется с t-тестом и формирует основу для практической рекомендации: ресурсы реактивационных кампаний следует концентрировать на сегменте с низким первым чеком, где располагается основной потенциал прироста retention.

## Сохранение результатов

Размеченные таблицы клиентов и матрица retention сохраняются для использования в ноутбуках 03, 04, 05 и 06.

In [ ]:
customers.to_parquet('../data/customers_labeled.parquet', index=False)
retention.to_parquet('../data/cohort_retention.parquet')
print('Сохранено:')
print('  data/customers_labeled.parquet')
print('  data/cohort_retention.parquet')

## Резюме

1. Когортная карта retention локализует основное падение в интервале первого месяца после первой покупки. Окно эффективного воздействия: первые тридцать дней.
2. Размер первого чека статистически значимо связан с вероятностью возврата (p << 0.001). Сегмент с чеком выше медианы возвращается заметно чаще.
3. Декабрьские когорты показывают сниженный retention, что соответствует структуре декабрьского спроса.

В ноутбуке 03 эти результаты переводятся в три продуктовые рекомендации с привязкой к измеримым метрикам.